<a href="https://colab.research.google.com/github/hemachandranrcs24-spec/BIS_LAB/blob/main/week31bm24cs115_hemachandranr_antcolony_shortest_computernetworkpath.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import random

# ==============================================================================
# PROBLEM SETUP: COMPUTER NETWORK SPECIFICATIONS
# Application: Network Routing (6 Routers: 0 to 5)
# Objective: Minimize combined end-to-end cost and delay
# ==============================================================================
num_routers = 6
source = 0
destination = 5
INF = float('inf')

# Define individual link costs
link_costs = np.array([
    [0,   3,   2,   INF, INF, INF],
    [3,   0,   1,   4,   INF, INF],
    [2,   1,   0,   6,   7,   INF],
    [INF, 4,   6,   0,   1,   5],
    [INF, INF, 7,   1,   0,   2],
    [INF, INF, INF, 5,   2,   0]
])

# Define individual link delays (in milliseconds)
link_delays = np.array([
    [0,   5,   1,   INF, INF, INF],
    [5,   0,   2,   6,   INF, INF],
    [1,   2,   0,   7,   8,   INF],
    [INF, 6,   7,   0,   2,   4],
    [INF, INF, 8,   2,   0,   2],
    [INF, INF, INF, 4,   2,   0]
])

# Combined Total Metric (Objective function per link: Cost + Delay)
# Links with infinity represent no physical communication link
total_link_metrics = np.where((link_costs == INF) | (link_delays == INF), INF, link_costs + link_delays)

# ==============================================================================
# ACO ROUTING ENGINE PARAMETERS
# ==============================================================================
num_ants = 15
max_iterations = 40
alpha = 1.0       # Pheromone sensitivity
beta = 2.5        # Heuristic sensitivity (highly favors low cost/delay links)
rho = 0.2         # Pheromone evaporation factor
Q = 20.0          # Pheromone deposit amount constant

# Initialize pheromone trails equally on all existing physical links
pheromones = np.where(total_link_metrics < INF, 1.0, 0.0)

# Heuristic Information (eta = 1 / total_link_metric)
with np.errstate(divide='ignore'):
    heuristic_info = 1.0 / total_link_metrics
    heuristic_info[total_link_metrics == INF] = 0
    np.fill_diagonal(heuristic_info, 0)

# Variables to track the optimal global routing solution
global_best_path = None
global_best_metric = INF

# ==============================================================================
# ACO ROUTING LOOP
# ==============================================================================
print("=" * 65)
print("  ANT COLONY OPTIMIZATION (ACO) COMPUTER NETWORK ROUTING ENGINE")
print("=" * 65)
print(f"Source Router:      {source}")
print(f"Destination Router: {destination}\n")

for iteration in range(max_iterations):
    all_paths = []
    all_metrics = []

    for ant in range(num_ants):
        current_router = source
        path = [current_router]
        visited = {current_router}
        is_trapped = False

        while current_router != destination:
            neighbors = []
            probabilities = []

            # Identify valid neighboring routers connected by links
            for next_router in range(num_routers):
                if total_link_metrics[current_router][next_router] < INF and next_router not in visited:
                    neighbors.append(next_router)

                    # Compute preference using pheromones and heuristic info
                    tau = pheromones[current_router][next_router] ** alpha
                    eta = heuristic_info[current_router][next_router] ** beta
                    probabilities.append(tau * eta)

            # If the ant runs into a dead end, abandon this routing path
            if not neighbors:
                is_trapped = True
                break

            # Select next router using probability distribution
            prob_sum = sum(probabilities)
            if prob_sum == 0:
                next_router = random.choice(neighbors)
            else:
                normalized_probs = [p / prob_sum for p in probabilities]
                next_router = random.choices(neighbors, weights=normalized_probs, k=1)[0]

            path.append(next_router)
            visited.add(next_router)
            current_router = next_router

        # If destination router is successfully reached, evaluate metrics
        if not is_trapped:
            total_metric = sum(total_link_metrics[path[i]][path[i+1]] for i in range(len(path)-1))
            all_paths.append(path)
            all_metrics.append(total_metric)

            if total_metric < global_best_metric:
                global_best_metric = total_metric
                global_best_path = path

    # Pheromone Update Phase
    if all_metrics:
        # 1. Evaporate pheromones
        pheromones *= (1 - rho)

        # 2. Deposit new pheromones based on path quality
        for path, metric in zip(all_paths, all_metrics):
            for i in range(len(path) - 1):
                u, v = path[i], path[i+1]
                pheromones[u][v] += Q / metric
                pheromones[v][u] += Q / metric

# ==============================================================================
# CALCULATE FINAL SEPARATED METRICS FOR THE OPTIMAL ROUTE
# ==============================================================================
final_cost = sum(link_costs[global_best_path[i]][global_best_path[i+1]] for i in range(len(global_best_path)-1))
final_delay = sum(link_delays[global_best_path[i]][global_best_path[i+1]] for i in range(len(global_best_path)-1))

# ==============================================================================
# VISUAL SHOWCASE OUTPUT
# ==============================================================================
print("=" * 65)
print("                     OPTIMIZATION COMPLETE")
print("=" * 65)
print(f"👉 Selected Path:          {' ➔ '.join(map(str, global_best_path))}")
print(f"📉 Total Combined Metric:   {global_best_metric}")
print(f"💰 End-to-End Link Cost:    {final_cost}")
print(f"⏳ End-to-End Link Delay:   {final_delay} ms")
print("=" * 65)


  ANT COLONY OPTIMIZATION (ACO) COMPUTER NETWORK ROUTING ENGINE
Source Router:      0
Destination Router: 5

                     OPTIMIZATION COMPLETE
👉 Selected Path:          0 ➔ 2 ➔ 3 ➔ 4 ➔ 5
📉 Total Combined Metric:   23.0
💰 End-to-End Link Cost:    11.0
⏳ End-to-End Link Delay:   12.0 ms
